# Player analysis

Set the `BRIDGE_PLAYER` environment variable to a BBO username and run `bridge-ai import data/` first. Outputs contain real hands, so they are stripped before commit (nbstripout); never commit an executed copy.

In [ ]:
import os

import matplotlib.pyplot as plt
import pandas as pd

from bridge_ai import analysis as an
from bridge_ai import ddbench as dd

PLAYER = os.environ["BRIDGE_PLAYER"]
df = pd.read_parquet("../.cache/hands.parquet")
len(df)

## Opening bids by seat and vulnerability

In [ ]:
op = an.openings(df, PLAYER)
top = op["call"].value_counts().head(8).index
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, by in zip(axes, ["seat_no", "vul"], strict=True):
    an.opening_table(op, by).loc[top].plot.bar(ax=ax)
    ax.set_title(f"Opening call share by {by}")
    ax.set_ylabel("share of openings")
plt.tight_layout()

## Likely psychs

Openings that miss a lenient standard definition (HCP and suit length). 2D is never flagged because it is often artificial.

In [ ]:
ps = an.psychs(op)
print(f"{len(ps)} flagged of {len(op)} openings")
ps["anomaly"].str.replace(r"\d+", "#", regex=True).value_counts().plot.barh()

## Opening leads

In [ ]:
ld = an.leads(df, PLAYER)
kinds = pd.crosstab(ld["kind"], ld["vs"], normalize="columns").round(2)
kinds.plot.bar(figsize=(10, 4))
ld.groupby("vs")[["is_trump", "is_bid_by_defence"]].mean().round(2)

## Contract and result patterns

In [ ]:
rp = an.result_patterns(df, PLAYER)
rp[rp.role == "declarer"].set_index("strain")[["made_pct"]].plot.bar(legend=False)
plt.ylabel("% contracts made as declarer")
rp

## Double-dummy and par benchmark

In [ ]:
ddf = dd.add_double_dummy(df)
pb = dd.player_benchmark(ddf, PLAYER)
dd.summary(pb)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
pb[pb.is_declarer]["tricks_vs_dd"].value_counts().sort_index().plot.bar(ax=axes[0])
axes[0].set_title("Declarer tricks vs double dummy")
pb["imps_vs_par"].plot.hist(bins=30, ax=axes[1])
axes[1].set_title("IMPs vs par")
plt.tight_layout()